### Бонусный проект Амангелдиевой Жамили

# 1. Построение модели

In [2]:
# Устанавливаем библиотеки
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import precision_score, recall_score, precision_recall_curve
from sklearn.metrics import precision_recall_curve
from sklearn.model_selection import GridSearchCV

import nltk
import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import SnowballStemmer
nltk.download('popular')
nltk.download('punkt')

[nltk_data] Downloading collection 'popular'
[nltk_data]    | 
[nltk_data]    | Downloading package cmudict to
[nltk_data]    |     C:\Users\Zhami\AppData\Roaming\nltk_data...
[nltk_data]    |   Package cmudict is already up-to-date!
[nltk_data]    | Downloading package gazetteers to
[nltk_data]    |     C:\Users\Zhami\AppData\Roaming\nltk_data...
[nltk_data]    |   Package gazetteers is already up-to-date!
[nltk_data]    | Downloading package genesis to
[nltk_data]    |     C:\Users\Zhami\AppData\Roaming\nltk_data...
[nltk_data]    |   Package genesis is already up-to-date!
[nltk_data]    | Downloading package gutenberg to
[nltk_data]    |     C:\Users\Zhami\AppData\Roaming\nltk_data...
[nltk_data]    |   Package gutenberg is already up-to-date!
[nltk_data]    | Downloading package inaugural to
[nltk_data]    |     C:\Users\Zhami\AppData\Roaming\nltk_data...
[nltk_data]    |   Package inaugural is already up-to-date!
[nltk_data]    | Downloading package movie_reviews to
[nltk_data]   

True

In [3]:
# Открываем файл токсичных комментариев
# Источник: https://www.kaggle.com/datasets/blackmoon/russian-language-toxic-comments

df = pd.read_csv("./train_comments.csv", sep=",")

In [5]:
df.shape

(14412, 2)

In [7]:
df.head()

,comment,toxic
0,"Верблюдов-то за что? Дебилы, бл...\n",1.0
1,"Хохлы, это отдушина затюканого россиянина, мол...",1.0
2,Собаке - собачья смерть\n,1.0
3,"Страницу обнови, дебил. Это тоже не оскорблени...",1.0
4,"тебя не убедил 6-страничный пдф в том, что Скр...",1.0


In [33]:
df.isna().sum()

comment    0
toxic      0
dtype: int64

In [8]:
df["toxic"] = df["toxic"].apply(int)
df.head()

,comment,toxic
0,"Верблюдов-то за что? Дебилы, бл...\n",1
1,"Хохлы, это отдушина затюканого россиянина, мол...",1
2,Собаке - собачья смерть\n,1
3,"Страницу обнови, дебил. Это тоже не оскорблени...",1
4,"тебя не убедил 6-страничный пдф в том, что Скр...",1


In [9]:
df["toxic"].value_counts()

toxic
0    9586
1    4826
Name: count, dtype: int64

In [13]:
# Выводим пару токсичных комментов
for i in df[df["toxic"] == 1]["comment"].head(5):
    print(i)

Верблюдов-то за что? Дебилы, бл...

Хохлы, это отдушина затюканого россиянина, мол, вон, а у хохлов еще хуже. Если бы хохлов не было, кисель их бы придумал.

Собаке - собачья смерть

Страницу обнови, дебил. Это тоже не оскорбление, а доказанный факт - не-дебил про себя во множественном числе писать не будет. Или мы в тебя верим - это ты и твои воображаемые друзья?

тебя не убедил 6-страничный пдф в том, что Скрипалей отравила Россия? Анализировать и думать пытаешься? Ватник что ли?)



In [12]:
# Выводим не токсичные комментарии
for i in df[df["toxic"] == 0]["comment"].head(5):
    print(i)

В шапке были ссылки на инфу по текущему фильму марвел. Эти ссылки были заменены на фразу Репортим брипидора, игнорируем его посты. Если этого недостаточно, чтобы понять, что модератор абсолютный неадекват, и его нужно лишить полномочий, тогда эта борда пробивает абсолютное дно по неадекватности.

Почитайте посты у этого автора,может найдете что нибудь полезное. Надеюсь помог) https: pikabu.ru story obyichnyie budni dezsluzhbyi 4932098

Про графику было обидно) я так то проходил все серии гта со второй части по пятую, кроме гта 4. И мне не мешала графика ни в одной из частей. На компе у меня было куча видеокарт. Начиная с 32мб RIVA TNT и заканчивая 2Гб 560Ti на которой я спокойно играю который год в танки, гта5, ведьмака3 купил на распродаже и начал проходить. Да, не на ультрах. С пониженными текстурами. И не мешает. Я не понимаю дрочева на графике, требовать графику уровня плойки 4 минимум. Мне надо чтобы глаза не резало, только и всего. По поводу управления, мне не хватает переходника

In [27]:
# Делим на train и test
X = df['comment']
y = df['toxic']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state=42)

In [28]:
y_train.value_counts()

toxic
0    7642
1    3887
Name: count, dtype: int64

In [29]:
y_test.value_counts()

toxic
0    1944
1     939
Name: count, dtype: int64

In [35]:
# Чистим предложения от окончаний, пункцуации, предлогов и союзов, чтоб правильно "кормить" модель


# sentence_example - исходный текст
sentence_example = df.iloc[1]["comment"]

# Разбиваем предложение на части
tokens = word_tokenize(sentence_example, language="russian")

# убираем пунктуации
tokens_without_punctuation = [i for i in tokens if i not in string.punctuation]

# убираем пунктуации и предлоги, союзы
russian_stop_words = stopwords.words("russian")
tokens_without_stop_words_and_punctuation = [i for i in tokens_without_punctuation if i not in russian_stop_words]

# Итог. Слова без окончании
snowball = SnowballStemmer(language="russian")
stemmed_tokens = [snowball.stem(i) for i in tokens_without_stop_words_and_punctuation]

print(f"Исходный текст: {sentence_example}")
print(f"Итог: {stemmed_tokens}")


Исходный текст: Хохлы, это отдушина затюканого россиянина, мол, вон, а у хохлов еще хуже. Если бы хохлов не было, кисель их бы придумал.

Итог: ['хохл', 'эт', 'отдушин', 'затюкан', 'россиянин', 'мол', 'вон', 'хохл', 'хуж', 'есл', 'хохл', 'кисел', 'придума']


In [36]:
# Заворачиваемм все это в функцию, чтобы потом применить на весь датасет

def tokenize_sentence(sentence: str, remove_stop_words: bool = True):
    tokens = word_tokenize(sentence, language="russian")
    tokens = [i for i in tokens if i not in string.punctuation]
    if remove_stop_words:
        tokens = [i for i in tokens if i not in russian_stop_words]
    tokens = [snowball.stem(i) for i in tokens]
    return tokens

In [37]:
# проверяем итог функции с предыдущим результатом
tokenize_sentence(sentence_example)

['хохл',
 'эт',
 'отдушин',
 'затюкан',
 'россиянин',
 'мол',
 'вон',
 'хохл',
 'хуж',
 'есл',
 'хохл',
 'кисел',
 'придума']

In [43]:
# с помощью vectorizer применяем для всех комментов в датафрейме
vectorizer = TfidfVectorizer(tokenizer=lambda x: tokenize_sentence(x, remove_stop_words=True))
features = vectorizer.fit_transform(X_train)

In [47]:
# строим модель
model = LogisticRegression(random_state=0)
model.fit(features, y_train)

LogisticRegression(random_state=0)

In [50]:
# Проверяем первый коммент
X_train.iloc[0]

'Я последнюю серию специально не смотрю, для меня этот сериал не закончился )\n'

In [51]:
# Вот результат, все ОК
model.predict(features[0])

array([0], dtype=int64)

In [52]:
# Чтобы в модель подавать приложениями, а не векторами, vectorizer запихаем в пайплайн
model_pipeline = Pipeline([
    ("vectorizer", TfidfVectorizer(tokenizer=lambda x: tokenize_sentence(x, remove_stop_words=True))),
    ("model", LogisticRegression(random_state=0))
])

In [53]:
# переобучаем
model_pipeline.fit(X_train, y_train)

C:\Users\Zhami\anaconda3\Lib\site-packages\sklearn\feature_extraction\text.py:525: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


Pipeline(steps=[('vectorizer',
                 TfidfVectorizer(tokenizer=<function <lambda> at 0x0000026365CF1800>)),
                ('model', LogisticRegression(random_state=0))])

In [54]:
# Проверка модели, результат 0
model_pipeline.predict(["Привет, как дела?"])

array([0], dtype=int64)

In [58]:
# Проверка модели, результат 1
model_pipeline.predict(["Отвали, скотина!"])

array([1], dtype=int64)

In [59]:
# precision_score
precision_score(y_test, y_pred=model_pipeline.predict(X_test))


0.8986384266263238

In [60]:
# recall_score
recall_score(y_test, y_pred=model_pipeline.predict(X_test))


0.6325878594249201

In [ ]:
# Все, модель готова :)

# 2. Парсинг комментариев под судом Бишимбаева с YouTube

In [76]:
# Импортируем библиотеки
from selenium import webdriver
from tqdm import tqdm
import time
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as ec
from selenium.webdriver.common import by


In [156]:
# Ссылки
url = 'https://www.youtube.com/'
url_video = 'https://www.youtube.com/watch?v=WED9XyRRuyA'
#comments = '//*[@id="content"]'
comments = '//div[@class="style-scope ytd-expander"]'

In [163]:
# Список для результата
scrapped = []

with webdriver.Chrome() as driver:
  
    wait = WebDriverWait(driver,1)
    driver.get(url_video)
    driver.maximize_window()
    time.sleep(5)

    # Количество прокруток 500 для сбора всех возможных комментариев
    for item in tqdm(range(250)): 
        wait.until(ec.visibility_of_element_located(
                        (by.By.TAG_NAME, "body")))\
                       .send_keys(webdriver.common.keys.Keys.END)
        time.sleep(2)
        
    for comment in wait.until(ec.presence_of_all_elements_located(
                              (by.By.CSS_SELECTOR, "#content-text"))):
      
        scrapped.append(comment.text)

# сохраняем результат в датафрейм
df_commetns = pd.DataFrame(scrapped)


100%|████████████████████████████████████████████████████████████████████████████████| 250/250 [08:42<00:00,  2.09s/it]


In [166]:
# в датафреймемы видим всего 1193 комментариев, когда на странице YouTube написано, что их больше 7,5к
# Проанализировала, под #content-text расположены только основные комментарии.
# А все ответы на этот комментарий под другими "адресами" находится.
# Нужно потом (когда будет время свободное) распарсить и эти ответы. Пока не получилось :)

df_commetns

,0
0,Меня бил первый муж. Я скрывала. Когда пришла ...
1,"Байжанов который не убивал, 10 раз попросил пр..."
2,Адвокатам Байжанова так противно сидеть рядом ...
3,"Игорь Вранчев это адвокат от Бога , почти все ..."
4,Удивительно почему Байжанову нельзя характериз...
...,...
1188,Я по началу не понимал почему казахи перевожят...
1189,Ну убил жену не специально же ну по закону...
1190,Свободу Байжанову
1191,"Я не защищаю Бишимбаева ,но она не уходила от ..."


In [167]:
# Вытаскиваем комменты поштучно, проверяем правильно ли записались
df_commetns.iloc[1][0]

'Байжанов который не убивал, 10 раз попросил прощения у близких Салтанат и сказал что готов понести наказание за бездействие. А этот упырь который жестоко убил, ни разу не извинился и даже вину не признает!'

In [168]:
df_commetns.shape[0]

1193

# 3. Применение модели на спарсенных данных.

In [170]:
# Метод df_commetns_pred = model_pipeline.predict(df_commetns_2) не сработал.
# Поэтому проверила на определенные комментарии, все ОК
# После все завернула в цикл

model_pipeline.predict([df_commetns.iloc[75][0]])

array([1], dtype=int64)

In [171]:
rresult = []
for i in tqdm(range(df_commetns.shape[0])): 
    res = model_pipeline.predict([df_commetns.iloc[i][0]])
    rresult.append(res) 


100%|█████████████████████████████████████████████████████████████████████████████| 1193/1193 [00:03<00:00, 321.21it/s]


In [172]:
# Записываем результат в датафрейм
df_commetns_done = df_commetns
df_commetns_done['pred'] = rresult


In [173]:
# Смотрим соотношение токсичных комментов на не_токсичные комменты
df_commetns_done['pred'].value_counts(normalize=True)

pred
[0]    0.856664
[1]    0.143336
Name: proportion, dtype: float64

In [175]:
# Пойманные токсичные комменты, первые 20 шт
for i in df_commetns_done[df_commetns_done["pred"] == 1][0].head(20):
    print(i)

Мать этого психопата, которого судят за убийство с особой жестокостью еще кого-то бессовестным называет. Они реально живут в своем мире. Вся семья неадекватная.
Допросите ясновидящую Бишимбаева!!! И восстановите запись с видеокамеры наблюдения которую удалили!!! Поднимите этот комментарий!!!!
Слава богу , переводчик нужен только одному участнику суда , а то бы мы пол суда не понели
Лицо матери Биша говорит само за себя, и когда он покупая цветы кричит скажите, что это для матери министра, будучи уже не министром жаль, что кроме бывшего громкого имени у него ничего нет, он пустой.
Мать Бишимбаева: «Бессовестный» по отношению к Байжанову 
️
Как у неё язык поворачивается что-либо говорить 
️
Лучше бы от стыда умирала и взгляд прятала !!!
Судья,прокурор Айжан и  защитники  Салты,вы  лучшие,Казахстан  может гордиться  вами!
🇰🇿🇰🇿🇰🇿🇰🇿🇰🇿🇰🇿🇰🇿🇰🇿🇰🇿
На байжанове нет лица
, ох, как он попал из за этого байского сынка
Уважаемый суд, давайте посадим его уже, да? 
️
Не для этого женщины мучаются прино

In [176]:
# Пойманные не_токсичные комменты, первые 20 шт# 
for i in df_commetns_done[df_commetns_done["pred"] == 0][0].head(20):
    print(i)

Меня бил первый муж. Я скрывала. Когда пришла к отцу летом в водолазке, пряча синяки. Он заставил ее снять. Увидел эту страсть . поехал забрал мои вещи и вещи моего ребёнка. И сказал, если я вернусь к этому уроду, у меня не будет отца. И чтоб я даже к нему после смерти не приходила на кладбище. Больше я не вернулась. Сейчас сыну 18 лет. Я вышла замуж , родила дочь. И каждый день в течении 15 лет, я говорю папе спасибо, что спас мне жизнь
Больше 2 тысяч лайков, и очень много добрых комментариев, про моего папу. Я плачу
, мне так приятно за моего папу, столько доброты в людях. Этот мир не безнадёжен 
Кому интересно, почему я не ушла раньше: муж начал истязать, после смерти мамы, у папы на руках были 3 младших брата. Я не могла себе позволить прийти со своими проблемами. Но, он спас меня, и 6 лет воспитывал моего сына, и кормил нас. Мамы нет уже , как 20 лет. Папа до сих пор не женился, помогает нам всем. И говорит, не женился, только потому что, хочет лежать с мамой в одной могиле 
. Мой

In [ ]:
# На этом все, спасибо!
# В будущем хочу более гибким сделать модель, чтоб он был более умным и больше выявлял. Посмотрим что получится :)